In [41]:
from tool_server.utils.utils import *
from tool_server.utils.prompts import tool_planning_model_prompt_one_tool_call
from tqdm import tqdm
import json, os

input_data = "/mnt/petrelfs/songmingyang/code/reasoning/data_construction/PixelReasoner/data/pixelreasonersft_groundingcrop_formatted.jsonl"

input_data = "/mnt/petrelfs/songmingyang/code/reasoning/data_construction/PixelReasoner/data/supply_grounding_data/pixelreasonersft_groundingcrop_sground.jsonl"
input_data2 = "/mnt/petrelfs/songmingyang/code/reasoning/data_construction/PixelReasoner/data/pixelreasonersft_crop_formatted.jsonl"

input_data = process_jsonl(input_data)
input_data2 = process_jsonl(input_data2)
output_path = "/mnt/petrelfs/songmingyang/code/reasoning/data_construction/PixelReasoner/data/sft_data/pixelreasonersft_groundingcrop.json"
output_path2 = "/mnt/petrelfs/songmingyang/code/reasoning/data_construction/PixelReasoner/data/sft_data/pixelreasonersft_crop.json"

pixel_reasoner_basedir = "/mnt/petrelfs/songmingyang/songmingyang/data/vl_reasoning/tool_dataset/PixelReasoner-SFT-Data"

tool_name_list = ["OCR","HighlightBox","GroundingDINO","DrawLine","LanguageModel","SegmentRegionAroundPoint","GetSubplotInfo","Crop","Point","DrawShape","MaskBox","GetBarInfo"]


In [43]:

def turn_into_sft_format(input_data):
    sft_data = []
    tool_statistic_dict = {}
    average_tool_call_list = []
    conv_unqualify = 0
    
    for item in tqdm(input_data):
        new_jpg_files = []
        conversations = item["response_messages"]
        origin_qid = item["origin_qid"]
        qid = item["qid"]
        
        image_dir_path = pixel_reasoner_basedir
        
        new_conversations = [{"from":"system","value":tool_planning_model_prompt_one_tool_call}]
        image_cnt = 0
        image_indexes = []
        terminate = False
        current_tool_call_num = 0
        # print(f"Processing conversation: {conversations}")
        if not isinstance(conversations, list):
            conversations = [conversations]
        if len(conversations) <= 2:
            continue
        
        for conv in conversations:
            # print(f"Processing conversation: {conv}")
            if "content" not in conv or "role" not in conv:
                terminate = True
                break
            if conv["role"] == "system":
                continue
            
            new_role = "gpt" if conv["role"] == "assistant" else "human"
            contents = conv["content"]
            
            new_value = ""
            
            for content in contents:
                # print(f"Processing content: {content}")

                if isinstance(content, str) or "text" in content:
                    if isinstance(content, str):
                        text_content = f"{content}"
                    elif "text" in content:
                        text_content = f"{content['text']}"
                    
                    text_content = text_content.replace("<image>","")
                    new_value += text_content
                    
                    if "<tool_call>" in text_content:
                        tool_json = text_content.split("<tool_call>")[-1].split("</tool_call>")[0]
                        
                        try:
                            tool_content = json.loads(tool_json)
                            tool_name = tool_content["name"]
                            
                            assert tool_name in tool_name_list, f"Tool name {tool_name} not in predefined list."
                            current_tool_call_num += 1
                            tool_statistic_dict[tool_name] = tool_statistic_dict.get(tool_name, 0) + 1
                            
                        except:
                            conv_unqualify += 1
                            tool_name = "invalid"
                            qualify = 0
                            terminate = True
                            break
                elif "image" in content or "image_url" in content:
                    new_value += f"\n<image>\n"
                    image_cnt += 1
                    image_content = "img_1"
                    if content.get("image_url"):
                        if isinstance(content["image_url"], str):
                            image_content = content["image_url"]
                        else:
                            image_content = content["image_url"].get("url", "img_1")
                    else:
                        image_content = content.get("image", "img_1")
                    
                    if not os.path.isabs(image_content):
                        image_path = os.path.join(image_dir_path, image_content)
                    else:
                        image_path = image_content
                        
                    if not os.path.exists(image_path):
                        terminate = True
                        break
                    
                    new_jpg_files.append(image_path)
                    

                elif "tool_code" in content or "tool_call" in content or "tool_response" in content or "tool_response_from" in content:
                    terminate = True
                    break
                else:
                    terminate = True
                    break
                    # raise ValueError(f"Unknown content type: {content['type']}")
            if terminate:
                break
            new_conversations.append({"from": new_role, "value": new_value})
        if terminate:
            continue
        average_tool_call_list.append(current_tool_call_num)
        
        assert len(new_jpg_files) == image_cnt, f"Image count mismatch: {len(new_jpg_files)} != {image_cnt}"
        
        new_item = {
            "qid": qid,
            "conversations": new_conversations,
            "images": new_jpg_files,
        }
        sft_data.append(new_item)
        
    average_tool_call = sum(average_tool_call_list) / len(average_tool_call_list)
    return {
        "sft_data": sft_data,
        "tool_statistic_dict": tool_statistic_dict,
        "average_tool_call": average_tool_call,
    }
        

In [44]:
data1_turn_res = turn_into_sft_format(input_data)

100%|██████████| 163/163 [00:00<00:00, 474.17it/s]


In [45]:
sft_data, tool_statistic_dict, average_tool_call = data1_turn_res["sft_data"], data1_turn_res["tool_statistic_dict"], data1_turn_res["average_tool_call"]
len(sft_data), tool_statistic_dict, average_tool_call,

(107, {'GroundingDINO': 86, 'Crop': 192, 'OCR': 14}, 2.6261682242990654)

In [46]:
write_json_file(sft_data, output_path)
len(sft_data)

107

In [47]:
data2_turn_res = turn_into_sft_format(input_data2)
sft_data2, tool_statistic_dict2, average_tool_call2 = data2_turn_res["sft_data"], data2_turn_res["tool_statistic_dict"], data2_turn_res["average_tool_call"]
len(sft_data2), tool_statistic_dict2, average_tool_call2,


100%|██████████| 400/400 [00:00<00:00, 694.66it/s]


(208,
 {'GroundingDINO': 93, 'Crop': 392, 'HighlightBox': 1, 'OCR': 19},
 1.9615384615384615)

In [48]:
write_json_file(sft_data2, output_path2)
len(sft_data2)

208